# Notebook 1: Setup and Data Preparation

**What this notebook does:**
1. Mounts Google Drive (to persist data and models across sessions)
2. Clones the parsing-mbert repository (which contains all the data)
3. Installs all required Python packages
4. Downloads UD 2.5 treebanks for Irish, Maltese, and Vietnamese
5. Verifies all data files are present and prints statistics

**Run this notebook FIRST before any other notebook.**

---
**Hardware**: Make sure you have a GPU connected in Colab.  
Go to: Runtime > Change runtime type > Hardware accelerator > T4 GPU

In [ ]:
# Step 1: Mount Google Drive
# Your models and data will be saved here and persist between sessions
from google.colab import drive
drive.mount('/content/drive')

# Create a workspace folder on Drive
import os
WORKSPACE = '/content/drive/MyDrive/thesis_mbert'
os.makedirs(WORKSPACE, exist_ok=True)
print(f'Workspace: {WORKSPACE}')

In [ ]:
# Step 2: Clone the repository
import os
REPO_URL = 'https://github.com/Sansgithub22/mtechthesis2Parsing-mbert.git'
REPO_DIR = '/content/parsing-mbert'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    print('Repository cloned.')
else:
    os.system(f'cd {REPO_DIR} && git pull')
    print('Repository already exists, pulled latest changes.')

# Add thesis source to Python path
import sys
THESIS_SRC = os.path.join(REPO_DIR, 'thesis', 'src')
if THESIS_SRC not in sys.path:
    sys.path.insert(0, THESIS_SRC)
print(f'Python path: {THESIS_SRC}')

In [ ]:
# Step 3: Install all required packages
# Colab already has PyTorch pre-installed — do NOT reinstall it

!pip install -q \
    transformers==4.38.0 \
    tokenizers==0.15.2 \
    datasets==2.18.0 \
    peft==0.10.0 \
    conllu==4.5.3 \
    accelerate==0.27.0 \
    sentencepiece==0.1.99 \
    seaborn \
    tqdm

import torch
print(f'PyTorch version already installed: {torch.__version__}')
print('All packages installed!')

In [ ]:
# Step 4: Download UD 2.5 treebanks
# We need Irish (IDT), Maltese (MUDT), Vietnamese (VTB)
# Singlish is already in the repo at data/sing/

import os

UD_DIR = os.path.join(WORKSPACE, 'ud_treebanks')
os.makedirs(UD_DIR, exist_ok=True)

# Download UD 2.5 from LINDAT
UD_URL = 'https://lindat.mff.cuni.cz/repository/xmlui/bitstream/handle/11234/1-3105/ud-treebanks-v2.5.tgz'

ud_archive = os.path.join(WORKSPACE, 'ud-treebanks-v2.5.tgz')
if not os.path.exists(ud_archive):
    print('Downloading UD 2.5 treebanks (~1GB)...')
    os.system(f'wget -q -O {ud_archive} "{UD_URL}"')
    print('Download complete.')
else:
    print('Archive already downloaded.')

# Extract
ud_extracted = os.path.join(WORKSPACE, 'ud-treebanks-v2.5')
if not os.path.exists(ud_extracted):
    print('Extracting...')
    os.system(f'tar -xzf {ud_archive} -C {WORKSPACE}')
    print('Extracted.')
else:
    print('Already extracted.')

In [ ]:
# Step 5: Set up the data paths we will use throughout all notebooks
# Run this cell and all the paths will be configured

import os
REPO_DIR = '/content/parsing-mbert'
WORKSPACE = '/content/drive/MyDrive/thesis_mbert'
UD_BASE = os.path.join(WORKSPACE, 'ud-treebanks-v2.5')

TREEBANKS = {
    'ga':   {
        'train': os.path.join(UD_BASE, 'UD_Irish-IDT', 'ga_idt-ud-train.conllu'),
        'dev':   os.path.join(UD_BASE, 'UD_Irish-IDT', 'ga_idt-ud-dev.conllu'),
        'test':  os.path.join(UD_BASE, 'UD_Irish-IDT', 'ga_idt-ud-test.conllu'),
    },
    'mt':   {
        'train': os.path.join(UD_BASE, 'UD_Maltese-MUDT', 'mt_mudt-ud-train.conllu'),
        'dev':   os.path.join(UD_BASE, 'UD_Maltese-MUDT', 'mt_mudt-ud-dev.conllu'),
        'test':  os.path.join(UD_BASE, 'UD_Maltese-MUDT', 'mt_mudt-ud-test.conllu'),
    },
    'vi':   {
        'train': os.path.join(UD_BASE, 'UD_Vietnamese-VTB', 'vi_vtb-ud-train.conllu'),
        'dev':   os.path.join(UD_BASE, 'UD_Vietnamese-VTB', 'vi_vtb-ud-dev.conllu'),
        'test':  os.path.join(UD_BASE, 'UD_Vietnamese-VTB', 'vi_vtb-ud-test.conllu'),
    },
    'sing': {
        'train': os.path.join(REPO_DIR, 'data', 'sing', 'train.conllu'),
        'dev':   os.path.join(REPO_DIR, 'data', 'sing', 'dev.conllu'),
        'test':  os.path.join(REPO_DIR, 'data', 'sing', 'test.conllu'),
    },
}

UNLABELED = {
    'ga':   os.path.join(REPO_DIR, 'data', 'unlabeled', 'ga'),
    'mt':   os.path.join(REPO_DIR, 'data', 'unlabeled', 'mt'),
    'vi':   os.path.join(REPO_DIR, 'data', 'unlabeled', 'vi'),
    'sing': os.path.join(REPO_DIR, 'data', 'unlabeled', 'sg'),
}

print('Data paths configured.')

In [ ]:
# Step 6: Verify all files exist and print statistics

import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'thesis', 'src'))

from data.conllu_reader import read_conllu, print_stats

LANG_NAMES = {'ga': 'Irish', 'mt': 'Maltese', 'vi': 'Vietnamese', 'sing': 'Singlish'}

print('=== Labeled Treebank Statistics ===')
for lang, paths in TREEBANKS.items():
    print(f'\n--- {LANG_NAMES[lang]} ({lang.upper()}) ---')
    for split, path in paths.items():
        if os.path.exists(path):
            sents = read_conllu(path)
            print_stats(sents, f'  {split}')
        else:
            print(f'  {split}: FILE NOT FOUND at {path}')

print('\n=== Unlabeled Data Statistics ===')
for lang, dir_path in UNLABELED.items():
    print(f'\n--- {LANG_NAMES[lang]} ({lang.upper()}) ---')
    for split in ['train', 'valid', 'test']:
        path = os.path.join(dir_path, f'{split}.txt')
        if os.path.exists(path):
            with open(path) as f:
                lines = [l for l in f if l.strip()]
            print(f'  {split}: {len(lines):,} sentences')
        else:
            print(f'  {split}: FILE NOT FOUND at {path}')

In [ ]:
# Step 7: Check GPU availability
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu}')
    print(f'GPU Memory: {mem:.1f} GB')
    print('Ready for training!')
else:
    print('WARNING: No GPU detected!')
    print('Go to Runtime > Change runtime type > GPU')

In [ ]:
# Step 8: Quick sanity check — tokenize a sentence with mBERT
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-multilingual-cased')

# Irish example
sentence = 'Ta Gaeilge ar an gclar.'
tokens = tokenizer.tokenize(sentence)
print(f'Irish sentence: {sentence}')
print(f'mBERT tokens:   {tokens}')
print(f'Unknown tokens: {[t for t in tokens if t == "[UNK]"]}')
print()

# Maltese example
sentence_mt = "Il-lingwa Maltija hija wahda unika."
tokens_mt = tokenizer.tokenize(sentence_mt)
print(f'Maltese sentence: {sentence_mt}')
print(f'mBERT tokens:     {tokens_mt}')
print(f'Unknown tokens:   {[t for t in tokens_mt if t == "[UNK]"]}')
print()
print('Setup complete! Proceed to Notebook 02.')